# 3. LightGBM
Tuned on the last training year with early stopping, then retrained on the full training window with five seeds.
Every tree is built on a random 80% of the training rows (row subsampling), so each seed gives a different model.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, matthews_corrcoef, accuracy_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

### round_trip: read the numbers back exactly as notebook 1 saved them ###
data = pd.read_csv("model_data.csv", parse_dates=["Date"], float_precision="round_trip")

FEATURES_A = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4", "ret_lag5",
              "hl_range", "intraday_ret", "dlog_volume"]
FEATURES_B = FEATURES_A + ["dist_sma10", "dist_sma50", "rsi14", "macd_rel", "macd_signal_rel"]
FEATURE_SETS = {"A": FEATURES_A, "B": FEATURES_B}

TEST_YEARS = [2020, 2021, 2022, 2023, 2024]

import itertools
import lightgbm as lgb

SEEDS = [0, 1, 2, 3, 4]

In [2]:
### the three evaluation settings: (setting, held-out unit, training stocks, test stocks) ###

TECH = ["AAPL", "MSFT", "AMZN", "GOOGL", "TSLA"]
ENERGY = ["XOM", "CVX", "COP"]
FOOD = ["PEP", "KO", "MDLZ"]
TECH_CROSS_SECTOR = ["AAPL", "MSFT", "GOOGL"]   # three tech stocks, so every sector has three

units = [("within", "all", TECH + ENERGY + FOOD, TECH + ENERGY + FOOD)]

for stock in TECH:
    units.append(("cross_stock", stock, [s for s in TECH if s != stock], [stock]))

units.append(("cross_sector", "Technology", ENERGY + FOOD, TECH_CROSS_SECTOR))
units.append(("cross_sector", "Energy", TECH_CROSS_SECTOR + FOOD, ENERGY))
units.append(("cross_sector", "Food & Beverage", TECH_CROSS_SECTOR + ENERGY, FOOD))

pd.DataFrame(units, columns=["setting", "held_out", "train_stocks", "test_stocks"])

,setting,held_out,train_stocks,test_stocks
0,within,all,"[AAPL, MSFT, AMZN, GOOGL, TSLA, XOM, CVX, COP,...","[AAPL, MSFT, AMZN, GOOGL, TSLA, XOM, CVX, COP,..."
1,cross_stock,AAPL,"[MSFT, AMZN, GOOGL, TSLA]",[AAPL]
2,cross_stock,MSFT,"[AAPL, AMZN, GOOGL, TSLA]",[MSFT]
3,cross_stock,AMZN,"[AAPL, MSFT, GOOGL, TSLA]",[AMZN]
4,cross_stock,GOOGL,"[AAPL, MSFT, AMZN, TSLA]",[GOOGL]
5,cross_stock,TSLA,"[AAPL, MSFT, AMZN, GOOGL]",[TSLA]
6,cross_sector,Technology,"[XOM, CVX, COP, PEP, KO, MDLZ]","[AAPL, MSFT, GOOGL]"
7,cross_sector,Energy,"[AAPL, MSFT, GOOGL, PEP, KO, MDLZ]","[XOM, CVX, COP]"
8,cross_sector,Food & Beverage,"[AAPL, MSFT, GOOGL, XOM, CVX, COP]","[PEP, KO, MDLZ]"


In [3]:
### train/test split with a one-day embargo ###

def remove_last_day(df):
    # the target of the last training day uses the first price of the next year
    return df[df["Date"] != df["Date"].max()]

def split(test_year, train_stocks, test_stocks):
    train = data[(data["year"] < test_year) & data["Ticker"].isin(train_stocks)]
    train = remove_last_day(train)
    test = data[(data["year"] == test_year) & data["Ticker"].isin(test_stocks)]
    return train, test

def metrics(y, proba):
    pred = (proba >= 0.5).astype(int)
    return {"auc": roc_auc_score(y, proba) if len(np.unique(proba)) > 1 else 0.5,
            "mcc": matthews_corrcoef(y, pred),
            "acc": accuracy_score(y, pred),
            "up_rate": pred.mean(),
            "recall_up": pred[y == 1].mean(),
            "recall_down": 1 - pred[y == 0].mean()}

In [4]:
### grid (contains the defaults: 31 leaves, 20 samples per leaf, learning rate 0.1, all features) ###
### row subsampling (80% of the rows per tree) is always on and is not part of the grid ###

grid = [{"num_leaves": nl, "min_child_samples": mcs, "learning_rate": lr, "feature_fraction": ff}
        for nl, mcs, lr, ff in itertools.product([7, 31], [20, 200], [0.02, 0.1], [0.7, 1.0])]
print(len(grid), "combinations")

def make_model(params, seed, n_trees=1000):
    # row subsampling is always on: every tree sees a random 80% of the training rows,
    # so each seed gives a different model
    return lgb.LGBMClassifier(n_estimators=n_trees, random_state=seed, feature_fraction_seed=seed,
                              bagging_fraction=0.8, bagging_freq=1, bagging_seed=seed,
                              n_jobs=2, verbose=-1, **params)

def fit_early_stopping(params, X, y, X_val, y_val):
    # at most 1,000 trees; stop when the validation ROC-AUC has not improved for 50 trees
    model = make_model(params, seed=0)
    model.fit(X, y, eval_set=[(X_val, y_val)], eval_metric="auc",
              callbacks=[lgb.early_stopping(50, verbose=False)])
    return model, max(model.best_iteration_, 1)

16 combinations


In [5]:
### train and test ###

results, predictions, importance = [], [], []

for test_year in TEST_YEARS:
    for setting, held_out, train_stocks, test_stocks in units:
        train, test = split(test_year, train_stocks, test_stocks)
        info = {"test_year": test_year, "setting": setting, "held_out": held_out,
                "n_train": len(train), "n_test": len(test)}

        for feature_set, features in FEATURE_SETS.items():

            ### tuning on the last training year, with early stopping ###
            val_year = test_year - 1
            fit_part = remove_last_day(train[train["year"] < val_year])
            val_part = train[train["year"] == val_year]
            best_params, best_trees, best_val_auc = None, None, -1
            for params in grid:
                model, n_trees = fit_early_stopping(params, fit_part[features], fit_part["Target"],
                                                    val_part[features], val_part["Target"])
                val_auc = roc_auc_score(val_part["Target"], model.predict_proba(val_part[features])[:, 1])
                if val_auc > best_val_auc:
                    best_params, best_trees, best_val_auc = params, n_trees, val_auc

            ### retrain on the full training window with five seeds ###
            probas = []
            for seed in SEEDS:
                model = make_model(best_params, seed, best_trees)
                model.fit(train[features], train["Target"])
                proba = model.predict_proba(test[features])[:, 1]
                probas.append(proba)
                train_auc = roc_auc_score(train["Target"], model.predict_proba(train[features])[:, 1])
                results.append({**info, "model": "lgbm", "features": feature_set, "seed": seed,
                                "params": str(best_params), "n_trees": best_trees,
                                "train_auc": train_auc, "val_auc": best_val_auc,
                                **metrics(test["Target"].to_numpy(), proba)})

                ### permutation importance (first seed) ###
                if seed == 0:
                    imp = permutation_importance(model, test[features], test["Target"], scoring="roc_auc",
                                                 n_repeats=5, random_state=0, n_jobs=1)
                    for feature, value in zip(features, imp.importances_mean):
                        importance.append({**info, "model": "lgbm", "features": feature_set,
                                           "feature": feature, "importance": value})

            predictions.append(test[["Date", "Ticker", "Target", "next_ret"]]
                               .assign(**info, model="lgbm", features=feature_set,
                                       proba=np.mean(probas, axis=0)))

    print(test_year, "done")

2020 done


2021 done


2022 done


2023 done


2024 done


In [6]:
### save ###

results = pd.DataFrame(results)
predictions = pd.concat(predictions, ignore_index=True)
importance = pd.DataFrame(importance)

results.to_csv("results/results_lgbm.csv", index=False)
predictions.to_csv("results/predictions_lgbm.csv.gz", index=False)
importance.to_csv("results/importance_lgbm.csv", index=False)

results.groupby(["setting", "features"])[["auc", "mcc", "acc", "up_rate"]].mean().round(3)

auc    mcc    acc  up_rate
setting      features                              
cross_sector A         0.512  0.022  0.522    0.847
             B         0.503  0.008  0.517    0.910
cross_stock  A         0.505  0.005  0.530    0.958
             B         0.506  0.003  0.531    0.960
within       A         0.498 -0.004  0.521    0.964
             B         0.500  0.006  0.524    0.944